
# **Tahap 3 Case Retrieval**

**Representasi vektor**

In [9]:
import os
import re
from google.colab import drive

drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords

file_path = "/content/drive/MyDrive/SUBCPMK3PENALARAN/Penalaran_Komputer/tahap_2/hasil_ekstraksi_putusan.csv"
df = pd.read_csv(file_path)

df['Dakwaan'] = df['Dakwaan'].fillna("").astype(str)
print("Sel 1 Selesai Data berhasil dibaca")

Sel 1 Selesai Data berhasil dibaca


In [11]:
nltk.download('stopwords')
stop_words_indo = stopwords.words('indonesian')
stop_words_indo.extend(['adanya', 'apabila', 'adalah', 'yg', 'dan', 'di', 'bahwa', 'kepada'])

vectorizer = TfidfVectorizer(stop_words=stop_words_indo)
X_tfidf = vectorizer.fit_transform(df['Dakwaan'])

print("Sel 2 Selesai Teks berhasil diubah menjadi angka (TF-IDF)")

Sel 2 Selesai Teks berhasil diubah menjadi angka (TF-IDF)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['baiknya', 'berkali', 'kali', 'kurangnya', 'mata', 'olah', 'sekurang', 'setidak', 'tama', 'tidaknya'] not in stop_words.
  warnings.warn(


In [12]:
X_train, X_test, df_train, df_test = train_test_split(
    X_tfidf, df,
    test_size=0.30,
    random_state=42
)

print("Sel 3 Selesai Data berhasil dibagi menjadi Train dan Test!")

Sel 3 Selesai Data berhasil dibagi menjadi Train dan Test!


In [13]:
kasus_baru_vektor = X_test[0]
info_kasus_baru = df_test.iloc[0]

print("=== KASUS BARU (QUERY) ===")
print(f"Nomor Perkara : {info_kasus_baru['Nomor Perkara']}")
print(f"Terdakwa      : {info_kasus_baru['Pihak Tergugat/Termohon/Terdakwa']}")
print("==========================\n")

skor_kemiripan = cosine_similarity(kasus_baru_vektor, X_train)[0]

df_hasil_pencarian = pd.DataFrame({
    'Nomor Perkara Kasus Lama': df_train['Nomor Perkara'].values,
    'Terdakwa Kasus Lama': df_train['Pihak Tergugat/Termohon/Terdakwa'].values,
    'Tingkat Kemiripan (%)': skor_kemiripan * 100
})

df_hasil_pencarian = df_hasil_pencarian.sort_values(by='Tingkat Kemiripan (%)', ascending=False)

print("=== 5 KASUS LAMA YANG PALING MIRIP ===")
print(df_hasil_pencarian.head(5).to_string(index=False))

=== KASUS BARU (QUERY) ===
Nomor Perkara : 27-k/pm.i-02/ad/ill/2025
Terdakwa      : dedy aprizal

=== 5 KASUS LAMA YANG PALING MIRIP ===
Nomor Perkara Kasus Lama Terdakwa Kasus Lama  Tingkat Kemiripan (%)
62-k/pm.ii-09/au/lv/2025      wildan mulyana                    0.0
65-k/pm.ii-09/ad/iv/2025         tomi faizin                    0.0
                 32-k/pm        arya prayogi                    0.0
  47-k/pm.i-04/al/v/2025            awaludin                    0.0
                      74   muhammad ramadhan                    0.0


In [14]:
from typing import List

def retrieve(query: str, k: int = 5) -> List[str]:
    """
    Mencari top-k kasus lama (case_id) yang paling mirip dengan query (teks kasus baru).
    """
    query_bersih = str(query)

    query_vektor = vectorizer.transform([query_bersih])

    skor_kemiripan = cosine_similarity(query_vektor, X_train)[0]

    df_hasil = pd.DataFrame({
        'case_id': df_train['Nomor Perkara'].values,
        'skor': skor_kemiripan
    })

    df_hasil_urut = df_hasil.sort_values(by='skor', ascending=False)

    top_k_case_id = df_hasil_urut['case_id'].head(k).tolist()

    return top_k_case_id

print("Sel 5 Selesai Fungsi retrieve() berhasil dibuat!")

Sel 5 Selesai Fungsi retrieve() berhasil dibuat!


In [15]:
import os
import json

queries_data = []

for i in range(5):
    teks_query_asli = df_test.iloc[i]['Dakwaan']
    nomor_perkara_query = df_test.iloc[i]['Nomor Perkara']

    top_5_hasil = retrieve(query=teks_query_asli, k=5)

    queries_data.append({
        "query_id": f"Q{i+1}",
        "query_case_id_asli": nomor_perkara_query,
        "query_text": teks_query_asli[:150] + "...",
        "ground_truth_case_id": top_5_hasil[:2]
    })

#json_path = os.path.join(output_dir, 'queries.json')
#with open(json_path, 'w', encoding='utf-8') as f:
    #json.dump(queries_data, f, indent=4, ensure_ascii=False)

print("Sel 6 Selesai: Tahap Pengujian dan Output Selesai!")
#print(f"File tersimpan di: {json_path}")
print("\nIsi dari queries.json (Cuplikan):")
print(json.dumps(queries_data[:2], indent=4))

Sel 6 Selesai: Tahap Pengujian dan Output Selesai!

Isi dari queries.json (Cuplikan):
[
    {
        "query_id": "Q1",
        "query_case_id_asli": "27-k/pm.i-02/ad/ill/2025",
        "query_text": "secara bersama-sama melakukan penipuan...",
        "ground_truth_case_id": [
            "62-k/pm.ii-09/au/lv/2025",
            "65-k/pm.ii-09/ad/iv/2025"
        ]
    },
    {
        "query_id": "Q2",
        "query_case_id_asli": "91-k/pm",
        "query_text": "desersi dimasa damai...",
        "ground_truth_case_id": [
            "32-k/pm",
            "36-k/pm.i-02/au/iv/2025"
        ]
    }
]


In [16]:

kata_kunci = "desersi"

list_nomor_perkara = retrieve(query=kata_kunci, k=5)

print(f"Kata Kunci Pencarian: '{kata_kunci}'\n")

for i, nomor in enumerate(list_nomor_perkara):
    data_kasus = df_train[df_train['Nomor Perkara'] == nomor].iloc[0]

    print(f"{i+1}. Rekomendasi Peringkat ke-{i+1}:")
    print(f"   - Nomor Perkara : {data_kasus['Nomor Perkara']}")
    print(f"   - Nama Terdakwa : {data_kasus['Pihak Tergugat/Termohon/Terdakwa']}")
    print(f"   - Jenis Perkara : {data_kasus['Jenis Perkara']}")
    print(f"   - Cuplikan Teks : {data_kasus['Dakwaan'][:100]}...")
    print("-" * 50)

Kata Kunci Pencarian: 'desersi'

1. Rekomendasi Peringkat ke-1:
   - Nomor Perkara : 18-k/pm.ii-11/ad/iv/2025
   - Nama Terdakwa : achmad sarif
   - Jenis Perkara : Pidana Militer
   - Cuplikan Teks : desersi dalam waktu damai...
--------------------------------------------------
2. Rekomendasi Peringkat ke-2:
   - Nomor Perkara : 61-k/pm.iii-12/ad/v/2025
   - Nama Terdakwa : mochammad amin
   - Jenis Perkara : Pidana Militer
   - Cuplikan Teks : desersi dalam waktu damai...
--------------------------------------------------
3. Rekomendasi Peringkat ke-3:
   - Nomor Perkara : 59-k/pm.i-02/ad/vii/2021
   - Nama Terdakwa : aguswanto
   - Jenis Perkara : Pidana Militer
   - Cuplikan Teks : desersi dalam waktu damai...
--------------------------------------------------
4. Rekomendasi Peringkat ke-4:
   - Nomor Perkara : 33-k/pm.i-04/ad/iii/2025
   - Nama Terdakwa : ign nova bayu asmara
   - Jenis Perkara : Pidana Militer
   - Cuplikan Teks : desersi dimasa damai...
------------------------